In [17]:
import os
from itertools import islice
from math import ceil
import numpy as np
import odl
from tqdm import tqdm
from skimage.transform import resize
from pydicom.filereader import dcmread
import h5py
import random

In [18]:
MU_WATER = 20
MU_AIR = 0.02
MU_MAX = 3071 * (MU_WATER - MU_AIR) / 1000 + MU_WATER

# Image physical size in meters
MIN_PT = [-0.13, -0.13]
MAX_PT = [0.13, 0.13]

# File containing list of accepted CT scan directories
DIR_LIST_FILE = "ct_scan_my.txt"

TRAIN_LIST_FILE = "ct_scan_my.txt"
VAL_LIST_FILE = "ct_scan_my.txt"
TEST_LIST_FILE = "ct_scan_my.txt"

DATA_PATH = "../data"
TARGET_PATH = "../data/ct_ellipses_new_dataset"

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

NUM_ANGLES = 256
RECO_IM_SHAPE = (128, 128)

# Image shape for simulation
IM_SHAPE = (250, 250)

# ASTRA implementation to use: "astra_cpu" or "astra_cuda"
IMPL = "astra_cuda"

# Whether to add Poisson noise to the simulated sinograms
ADD_NOISE = False

PHOTONS_PER_PIXEL = 4096

# Number of samples stored in each HDF5 file
NUM_SAMPLES_PER_FILE = 128

In [ ]:
def load_files_from_dir_list(dir_list_file):
    """Load list of files from directories specified in DIR_LIST_FILE."""
    all_files = []

    # Load list of directories
    with open(dir_list_file, "r") as f:
        dir_list = [line.strip() for line in f if line.strip()]

    for rel_path in dir_list:
        abs_path = os.path.join(DATA_PATH, rel_path)

        if not os.path.isdir(abs_path):
            print(f"Warning: directory does not exist: {abs_path}")
            continue

        files = os.listdir(abs_path)

        for f in files:
            if f[0] == "I" or f.endswith(".dcm"):
                all_files.append(os.path.join(abs_path, f))

    return all_files

os.makedirs(TARGET_PATH, exist_ok=True)

train_files = load_files_from_dir_list(TRAIN_LIST_FILE)
val_files = load_files_from_dir_list(VAL_LIST_FILE)
test_files = load_files_from_dir_list(TEST_LIST_FILE)

# Shuffle the file lists
random.shuffle(train_files)
random.shuffle(val_files)
random.shuffle(test_files)

file_list = {
    "train": train_files,
    "validation": val_files,
    "test": test_files
}

print(f"Number of training files: {len(train_files)}")
print(f"Number of validation files: {len(file_list["validation"])}")
print(f"Number of test files: {len(file_list["test"])}")

Number of training files: 100
Number of validation files: 0
Number of test files: 0


In [12]:
def lidc_idri_gen(part="train"):
    seed = 0
    if part == "validation":
        seed = 1
    elif part == "test":
        seed = 2
    r = np.random.RandomState(seed)
    for dcm_file in file_list[part]:
        dataset = dcmread(os.path.join(dcm_file))
        array = dataset.pixel_array.astype(np.float32).T

        # resize image to RECO_IM_SHAPE
        array = resize(array, RECO_IM_SHAPE, order=1)

        # rescale by dicom meta info
        array *= dataset.RescaleSlope
        array += dataset.RescaleIntercept

        # add noise to get continuous values from discrete ones
        array += r.uniform(0.0, 1.0, size=array.shape)

        # convert values
        array *= (MU_WATER - MU_AIR) / 1000
        array += MU_WATER
        array /= MU_MAX
        np.clip(array, 0.0, 1.0, out=array)

        yield array

In [13]:
lidc_idri_gen_len = {p: len(file_list[p]) for p in ["train", "validation", "test"]}

In [14]:
reco_space = odl.uniform_discr(
    min_pt=MIN_PT,
    max_pt=MAX_PT,
    shape=RECO_IM_SHAPE,
    dtype=np.float32,
)
space = odl.uniform_discr(
    min_pt=MIN_PT, max_pt=MAX_PT, shape=IM_SHAPE, dtype=np.float64
)

reco_geometry = odl.tomo.parallel_beam_geometry(
    reco_space, num_angles=NUM_ANGLES
)
geometry = odl.tomo.parallel_beam_geometry(
    space,
    num_angles=NUM_ANGLES,
    det_shape=reco_geometry.detector.shape,
)

reco_ray_trafo = odl.tomo.RayTransform(
    reco_space, reco_geometry, impl=IMPL
)
ray_trafo = odl.tomo.RayTransform(space, geometry, impl=IMPL)

print(ray_trafo.range.shape)
print(reco_space.shape)

rs = np.random.RandomState(3)

(256, 183)
(128, 128)


In [15]:
def forward_fun(im):
    # upsample ground_truth in each dimension
    # before application of forward operator in order to avoid
    # inverse crime
    im_resized = resize(im * MU_MAX, IM_SHAPE, order=1)

    # apply forward operator
    data = ray_trafo(im_resized).asarray()

    data *= -1
    np.exp(data, out=data)
    data *= PHOTONS_PER_PIXEL
    
    return data

In [16]:
for part in ["train", "validation", "test"]:
    gen = lidc_idri_gen(part)
    n_files = ceil(lidc_idri_gen_len[part] / NUM_SAMPLES_PER_FILE)
    for filenumber in tqdm(range(n_files), desc=part):
        obs_filename = os.path.join(
            TARGET_PATH, "observation_{}_{:04d}.hdf5".format(part, filenumber)
        )
        ground_truth_filename = os.path.join(
            TARGET_PATH, "ground_truth_{}_{:04d}.hdf5".format(part, filenumber)
        )
        with h5py.File(obs_filename, "w") as observation_file, h5py.File(
            ground_truth_filename, "w"
        ) as ground_truth_file:
            observation_dataset = observation_file.create_dataset(
                "data",
                shape=(NUM_SAMPLES_PER_FILE,) + ray_trafo.range.shape,
                maxshape=(NUM_SAMPLES_PER_FILE,) + ray_trafo.range.shape,
                dtype=np.float32,
                chunks=True,
            )
            ground_truth_dataset = ground_truth_file.create_dataset(
                "data",
                shape=(NUM_SAMPLES_PER_FILE,) + reco_space.shape,
                maxshape=(NUM_SAMPLES_PER_FILE,) + reco_space.shape,
                dtype=np.float32,
                chunks=True,
            )

            im_buf = [im for im in islice(gen, NUM_SAMPLES_PER_FILE)]
            data_buf = [forward_fun(im) for im in im_buf]

            for i, (im, data) in enumerate(zip(im_buf, data_buf)):
                if ADD_NOISE:
                    data = rs.poisson(data)
                data = data / PHOTONS_PER_PIXEL
                np.maximum(0.1 / PHOTONS_PER_PIXEL, data, out=data)
                np.log(data, out=data)
                data /= -MU_MAX
                observation_dataset[i] = data
                ground_truth_dataset[i] = im

            # resize last file
            if filenumber == n_files - 1:
                observation_dataset.resize(
                    lidc_idri_gen_len[part] - (n_files - 1) * NUM_SAMPLES_PER_FILE,
                    axis=0,
                )
                ground_truth_dataset.resize(
                    lidc_idri_gen_len[part] - (n_files - 1) * NUM_SAMPLES_PER_FILE,
                    axis=0,
                )

train: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]
validation: 0it [00:00, ?it/s]
test: 0it [00:00, ?it/s]
